# 06 — Fused Attention

**Ladder:** Full $QK^{\top}$ (naive) → blocked tiles + **online softmax** → Triton fused forward (never store $N\times N$).

**Study tip.** Memorize the online-softmax rescale when the running max increases. Naive $S$ is $O(N^2)$ memory — that is the bug fusion fixes.

Official: [Fused Attention](https://triton-lang.org/main/getting-started/tutorials/06-fused-attention.html)

Figures: [`docs/images/`](docs/images/) · guide: [`docs/visual-guide.md`](docs/visual-guide.md)

---

## 0. Scaled dot-product attention

For $Q,K,V \in \mathbb{R}^{N \times D}$ (one head) and scale $\alpha = 1/\sqrt{D}$:

$$
S = \alpha\, Q K^{\top} \in \mathbb{R}^{N \times N},
\qquad
P = \mathrm{softmax}_{\mathrm{row}}(S),
\qquad
O = P V \in \mathbb{R}^{N \times D}.
$$

**Memory.** Materializing $S$ (or $P$) costs $\Theta(N^{2})$ — the FlashAttention idea is to never store $S$ globally; stream $K,V$ tiles and keep running softmax statistics.

![Whiteboard: full $QK^	op$ vs blocked online-softmax](docs/images/benchmark-attention.png)

*Theory board — $O(N^2)$ score matrix vs online-softmax tiles that never store full $S$.*

**Teaching sizes.** $N=128$, $D=64$, `BLOCK_M=BLOCK_N=32` — four $Q$-tiles, each streaming four $K/V$ tiles.

### Theory board

![Whiteboard: full $QK^\top$ vs blocked online-softmax](docs/images/whiteboard-theory.jpg)

*Theory board — $O(N^2)$ score matrix vs online-softmax tiles that never store full $S$.*

### Pitfalls

- Materializing S=QKᵀ is O(N²) memory — blows up for long context.
- Online-softmax must rescaled previous acc when m updates.
- Wrong scale α=1/√D changes attention sharpness.

### Self-check

1. State the online update for m, ℓ, acc.
2. Why never store full S in the fused kernel?
3. Complexity of memory: classic vs Flash-style.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import math

import pytest
import torch
import triton
import triton.language as tl

def _active_device():
    """Pick Triton active device, else CUDA, else CPU (study helper; no common/ package)."""
    try:
        import triton

        try:
            return triton.runtime.driver.active.get_active_torch_device()
        except Exception:
            pass
    except Exception:
        pass
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


DEVICE = _active_device()


def require_gpu() -> None:
    if not torch.cuda.is_available() and getattr(DEVICE, "type", None) != "cuda":
        pytest.skip(f"GPU not available (active device: {DEVICE})")



## 1. Online softmax — update equations

Process key blocks $t = 1,2,\ldots$ with scores $s^{(t)}$ (for a fixed $Q$-tile) and values $V^{(t)}$. Maintain running

$$
m^{(t)} \in \mathbb{R}^{B_M},\quad
\ell^{(t)} \in \mathbb{R}^{B_M},\quad
\mathrm{Acc}^{(t)} \in \mathbb{R}^{B_M \times D}.
$$

**Initialize:** $m^{(0)}=-\infty$, $\ell^{(0)}=0$, $\mathrm{Acc}^{(0)}=0$.

**Update (row-wise):**

$$
\begin{aligned}
m^{(t)} &= \max\bigl(m^{(t-1)},\, \max_j s^{(t)}_{:\,j}\bigr), \\[4pt]
\ell^{(t)} &= e^{m^{(t-1)}-m^{(t)}}\,\ell^{(t-1)} + \sum_j \exp\bigl(s^{(t)}_{:\,j}-m^{(t)}\bigr), \\[4pt]
\mathrm{Acc}^{(t)} &= e^{m^{(t-1)}-m^{(t)}}\,\mathrm{Acc}^{(t-1)} + \exp\bigl(s^{(t)}-m^{(t)}\bigr)\, V^{(t)}.
\end{aligned}
$$

**Finalize:** after the last block $T$,

$$
O_{\mathrm{tile}} = \frac{\mathrm{Acc}^{(T)}}{\ell^{(T)}}.
$$

**Claim.** $O_{\mathrm{tile}}$ equals $\mathrm{softmax}(S)\,V$ on that query tile (up to FP error).

*Idea.* Softmax is shift-invariant. When the running max increases, rescale previous $\ell$ and $\mathrm{Acc}$ by $e^{m_{\mathrm{old}}-m_{\mathrm{new}}}$ so all mass is expressed under the new max — same algebra as concatenating score blocks and doing one softmax.


## 2. Naive — materialize full $S$

$$
S = \alpha Q K^{\top},\qquad
P = \mathrm{softmax}_{\mathrm{row}}(S),\qquad
O = P V.
$$

Implemented by `naive_attention_pseudo`. Clear reference; allocates $\Theta(N^{2})$.


In [ ]:
def naive_attention_pseudo(Q, K, V):
    """
    Standard scaled-dot-product attention for one (or batched) head:
    (See markdown cells above for the math.)
    """
    D = Q.shape[-1]
    scale = 1.0 / math.sqrt(D)

    # Full QK^T — allocates an N×N tensor (bad for long sequences)
    S = torch.matmul(Q, K.transpose(-2, -1)) * scale
    P = torch.softmax(S, dim=-1)
    O = torch.matmul(P, V)
    return O


## 3. Intermediate — blocked + online softmax (Python)

For each $Q$-tile of height $B_M$, loop $K/V$-tiles of height $B_N$:

$$
s^{(t)} = \alpha\, Q_{\mathrm{tile}} (K^{(t)})^{\top},
\quad\text{then apply the §1 updates with } V^{(t)}.
$$

Implemented by `blocked_attention_pseudo`. Scratch is $O(B_M B_N + B_M D)$, not $O(N^{2})$.

| Stage | Function | Stores full $S$? |
|-------|----------|------------------|
| Naive | `naive_attention_pseudo` | yes |
| Blocked online | `blocked_attention_pseudo` | no |
| Triton | `_attn_fwd_kernel` / `attention` | no |


In [ ]:
def blocked_attention_pseudo(Q, K, V, BLOCK_M=32, BLOCK_N=32):
    """
    Same math as naive, but process query rows in BLOCK_M tiles and key/value
    (See markdown cells above for the math.)
    """
    *batch, N, D = Q.shape
    assert N % BLOCK_M == 0 and N % BLOCK_N == 0, "assume divisible for teaching"

    scale = 1.0 / math.sqrt(D)
    O = torch.zeros_like(Q)

    # Flatten leading batch dims so we can loop over heads/batches simply
    Qf = Q.reshape(-1, N, D)
    Kf = K.reshape(-1, N, D)
    Vf = V.reshape(-1, N, D)
    Of = O.reshape(-1, N, D)

    for b in range(Qf.shape[0]):
        for start_m in range(0, N, BLOCK_M):
            q = Qf[b, start_m : start_m + BLOCK_M, :]  # [BLOCK_M, D]
            # Accumulate attention over key blocks with online softmax
            m_i = torch.full((BLOCK_M,), -float("inf"), dtype=torch.float32, device=Q.device)
            l_i = torch.zeros(BLOCK_M, dtype=torch.float32, device=Q.device)
            acc = torch.zeros(BLOCK_M, D, dtype=torch.float32, device=Q.device)

            for start_n in range(0, N, BLOCK_N):
                k = Kf[b, start_n : start_n + BLOCK_N, :]  # [BLOCK_N, D]
                v = Vf[b, start_n : start_n + BLOCK_N, :]  # [BLOCK_N, D]
                qk = (q.float() @ k.float().T) * scale  # [BLOCK_M, BLOCK_N]

                m_ij = qk.max(dim=-1).values
                m_new = torch.maximum(m_i, m_ij)
                alpha = torch.exp(m_i - m_new)
                p = torch.exp(qk - m_new[:, None])
                l_i = l_i * alpha + p.sum(dim=-1)
                m_i = m_new
                acc = acc * alpha[:, None] + p @ v.float()

            Of[b, start_m : start_m + BLOCK_M, :] = (acc / l_i[:, None]).to(Q.dtype)

    return O


## 4. Triton fused forward

Same online-softmax algebra in SRAM: load $Q$/$K$/$V$ tiles, `tl.dot` for scores, update $(m,\ell,\mathrm{Acc})$, store $O$ only.

Optional causal mask: $S_{ij}=-\infty$ for $j>i$ (or skip pure-future $K$-tiles). Speedups vs naive are **IO** (fewer HBM trips), not fewer FLOPs — both are $\Theta(N^{2} D)$.


In [ ]:
@triton.jit
def _attn_fwd_kernel(
    Q,
    K,
    V,
    Out,
    stride_qz,
    stride_qh,
    stride_qm,
    stride_qk,
    stride_kz,
    stride_kh,
    stride_kn,
    stride_kk,
    stride_vz,
    stride_vh,
    stride_vn,
    stride_vk,
    stride_oz,
    stride_oh,
    stride_om,
    stride_ok,
    Z,
    H,
    N_CTX,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_DMODEL: tl.constexpr,
):
    """One program = one query tile of BLOCK_M rows for one (batch, head)."""
    start_m = tl.program_id(0)
    off_hz = tl.program_id(1)
    off_z = off_hz // H
    off_h = off_hz % H

    qvk_offset = off_z.to(tl.int64) * stride_qz + off_h.to(tl.int64) * stride_qh

    # Load Q tile once; stream K/V tiles along the sequence
    offs_m = start_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_d = tl.arange(0, BLOCK_DMODEL)
    q_ptrs = Q + qvk_offset + offs_m[:, None] * stride_qm + offs_d[None, :] * stride_qk
    q_mask = offs_m[:, None] < N_CTX
    q = tl.load(q_ptrs, mask=q_mask, other=0.0)

    acc = tl.zeros([BLOCK_M, BLOCK_DMODEL], dtype=tl.float32)
    m_i = tl.zeros([BLOCK_M], dtype=tl.float32) - float("inf")
    l_i = tl.zeros([BLOCK_M], dtype=tl.float32)
    scale = 1.0 / tl.sqrt(tl.cast(BLOCK_DMODEL, tl.float32))

    for start_n in range(0, N_CTX, BLOCK_N):
        start_n = tl.multiple_of(start_n, BLOCK_N)
        offs_n = start_n + tl.arange(0, BLOCK_N)

        k_ptrs = K + qvk_offset + offs_n[:, None] * stride_kn + offs_d[None, :] * stride_kk
        k_mask = offs_n[:, None] < N_CTX
        k = tl.load(k_ptrs, mask=k_mask, other=0.0)

        qk = tl.dot(q, tl.trans(k)) * scale
        # Mask padded key positions so they never win the max / get weight
        qk = tl.where(offs_n[None, :] < N_CTX, qk, float("-inf"))

        m_ij = tl.max(qk, axis=1)
        m_new = tl.maximum(m_i, m_ij)
        alpha = tl.exp(m_i - m_new)
        p = tl.exp(qk - m_new[:, None])
        l_i = l_i * alpha + tl.sum(p, axis=1)
        m_i = m_new

        v_ptrs = V + qvk_offset + offs_n[:, None] * stride_vn + offs_d[None, :] * stride_vk
        v = tl.load(v_ptrs, mask=k_mask, other=0.0)
        acc = acc * alpha[:, None] + tl.dot(p.to(v.dtype), v)

    acc = acc / l_i[:, None]
    o_ptrs = Out + qvk_offset + offs_m[:, None] * stride_om + offs_d[None, :] * stride_ok
    tl.store(o_ptrs, acc.to(Out.dtype.element_ty), mask=q_mask)


def attention(q, k, v, BLOCK_M=64, BLOCK_N=64):
    """Launch fused forward attention. q/k/v: [Z, H, N_CTX, D]."""
    Z, H, N_CTX, D = q.shape
    assert k.shape == v.shape == (Z, H, N_CTX, D)
    out = torch.empty_like(q)
    grid = (triton.cdiv(N_CTX, BLOCK_M), Z * H)
    _attn_fwd_kernel[grid](
        q,
        k,
        v,
        out,
        q.stride(0),
        q.stride(1),
        q.stride(2),
        q.stride(3),
        k.stride(0),
        k.stride(1),
        k.stride(2),
        k.stride(3),
        v.stride(0),
        v.stride(1),
        v.stride(2),
        v.stride(3),
        out.stride(0),
        out.stride(1),
        out.stride(2),
        out.stride(3),
        Z,
        H,
        N_CTX,
        BLOCK_M=BLOCK_M,
        BLOCK_N=BLOCK_N,
        BLOCK_DMODEL=D,
    )
    return out


# Alias used in docs / older snippets
attention_ref = naive_attention_pseudo


In [ ]:
def test_pseudo_matches_torch() -> None:
    torch.manual_seed(0)
    Z, H, N, D = 1, 2, 64, 32
    q = torch.randn(Z, H, N, D)
    k = torch.randn(Z, H, N, D)
    v = torch.randn(Z, H, N, D)
    ref = naive_attention_pseudo(q, k, v)
    blocked = blocked_attention_pseudo(q, k, v, BLOCK_M=32, BLOCK_N=32)
    assert torch.allclose(blocked, ref, atol=1e-4, rtol=1e-4)


def test_correctness() -> None:
    require_gpu()
    torch.manual_seed(0)
    Z, H, N_CTX, D = 1, 2, 128, 64
    q = torch.randn(Z, H, N_CTX, D, device=DEVICE, dtype=torch.float16)
    k = torch.randn(Z, H, N_CTX, D, device=DEVICE, dtype=torch.float16)
    v = torch.randn(Z, H, N_CTX, D, device=DEVICE, dtype=torch.float16)
    ref = naive_attention_pseudo(q.float(), k.float(), v.float()).half()
    out = attention(q, k, v, BLOCK_M=64, BLOCK_N=64)
    assert torch.allclose(out, ref, atol=2e-2, rtol=2e-2)
    print(f"fused attention OK  shape={tuple(out.shape)}")
